# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Token loaded and login successful!")

Token loaded and login successful!


In [ ]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
files = fs.ls("datasets/FlyRank/internship-warehouse", detail=False)
for f in files:
    print(f)



files = fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance", detail=False)
for f in files:
    print(f)



datasets/FlyRank/internship-warehouse/fact_content_daily_performance
datasets/FlyRank/internship-warehouse/.gitattributes
datasets/FlyRank/internship-warehouse/README.md
datasets/FlyRank/internship-warehouse/dim_clients.parquet
datasets/FlyRank/internship-warehouse/dim_content.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06
datasets/FlyRank/internship-warehouse/fact_content_daily_perfor

In [ ]:
files = fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03", detail=False)
for f in files:
    print(f)

datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [2]:
import pandas as pd

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_march = pd.read_parquet(url)

print("Shape:", df_march.shape)
print("Date range:", df_march["report_date"].min(), "to", df_march["report_date"].max())
df_march.head()

print(df_march.columns.tolist())

Shape: (9841378, 31)
Date range: 2026-03-01 to 2026-03-31
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In fact_content_daily_performance, one row represents one page, for one client, on one specific day — not a single overall snapshot like in the starter CSV. I proved this by checking one page (content_id: content_b7e512995f79d5a6) for the client client_73cda7b4e4f265ea across March 2026: it had exactly 31 rows, one for each day of the month. This confirms the table works like a daily diary — every page gets a fresh entry each day it's tracked, rather than one lifetime total. This matters for my lane because any feature I build (like average CTR or engagement) needs to be calculated across these daily rows for a page, not read off a single row.

In [ ]:
# Pick ONE page and see how many rows it has in March
sample_content = df_march["content_hash_id"].iloc[0]
sample_client = df_march["client_hash_id"].iloc[0]

rows_for_this_page = df_march[
    (df_march["content_hash_id"] == sample_content) &
    (df_march["client_hash_id"] == sample_client)
]

print("Page ID:", sample_content)
print("Number of rows for this one page in March:", rows_for_this_page.shape[0])
rows_for_this_page[["report_date", "client_hash_id", "content_hash_id"]]

Page ID: content_b7e512995f79d5a6
Number of rows for this one page in March: 31


,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
555866,2026-03-03,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1334822,2026-03-04,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1499377,2026-03-05,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1573240,2026-03-07,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1594884,2026-03-02,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2009723,2026-03-08,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2281373,2026-03-06,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2439321,2026-03-09,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2553195,2026-03-10,client_73cda7b4e4f265ea,content_b7e512995f79d5a6


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I will use fact_content_daily_performance (partitioned by month, using March 2026) as my main table, it has all the behavior data I need (clicks, impressions, sessions). I will also check dim_clients briefly, just to confirm that March 2026 is a valid, fully-tracked month for the clients in my data (not a month before their tracking started), so I don't mistake missing data for real zeros.

In [7]:
# Check: how many clients had already started being tracked before March 2026?
import pandas as pd



url_clients = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"
df_clients = pd.read_parquet(url_clients)

print("Shape:", df_clients.shape)
print("Columns:", df_clients.columns.tolist())
df_clients.head()

import datetime

march_start = datetime.date(2026, 3, 1)

gsc_ready = (df_clients["gsc_data_start"] <= march_start).sum()
ga4_ready = (df_clients["ga4_data_start"] <= march_start).sum()

print("Total clients:", df_clients.shape[0])
print("Clients with GSC data available by March 2026:", gsc_ready)
print("Clients with GA4 data available by March 2026:", ga4_ready)
print("\nMissing gsc_data_start:", df_clients["gsc_data_start"].isna().sum())
print("Missing ga4_data_start:", df_clients["ga4_data_start"].isna().sum())

print("\nEarliest / latest gsc_data_start:", df_clients["gsc_data_start"].dropna().min(), "/", df_clients["gsc_data_start"].dropna().max())
print("Earliest / latest ga4_data_start:", df_clients["ga4_data_start"].dropna().min(), "/", df_clients["ga4_data_start"].dropna().max())

ga4_ready_clients = df_clients[df_clients["ga4_data_start"] <= march_start]
earliest_ga4_start_relevant = ga4_ready_clients["ga4_data_start"].min()

print(f"Clients with GA4 ready by March 2026: {len(ga4_ready_clients)}")
print(f"Earliest GA4 start among THOSE clients: {earliest_ga4_start_relevant}")

Shape: (104, 9)
Columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
Total clients: 104
Clients with GSC data available by March 2026: 52
Clients with GA4 data available by March 2026: 26

Missing gsc_data_start: 37
Missing ga4_data_start: 53

Earliest / latest gsc_data_start: 2025-01-27 / 2026-06-02
Earliest / latest ga4_data_start: 2025-10-29 / 2026-06-01
Clients with GA4 ready by March 2026: 26
Earliest GA4 start among THOSE clients: 2025-10-29


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# Query 3: Availability check on March 2026 data
import pandas as pd

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_march = pd.read_parquet(url)

print("Shape:", df_march.shape)
print("Date range:", df_march["report_date"].min(), "to", df_march["report_date"].max())


total_rows_march = len(df_march)

gsc_available_rows = df_march[df_march["gsc_data_available"] == True].shape[0]
ga4_available_rows = df_march[df_march["ga4_data_available"] == True].shape[0]

print("Total rows in March 2026:", total_rows_march)
print("Rows where gsc_data_available IS TRUE:", gsc_available_rows)
print("Rows where ga4_data_available IS TRUE:", ga4_available_rows)
print("Percent GSC-available:", round(gsc_available_rows / total_rows_march * 100, 1), "%")
print("Percent GA4-available:", round(ga4_available_rows / total_rows_march * 100, 1), "%")

Shape: (9841378, 31)
Date range: 2026-03-01 to 2026-03-31
Total rows in March 2026: 9841378
Rows where gsc_data_available IS TRUE: 3611061
Rows where ga4_data_available IS TRUE: 413966
Percent GSC-available: 36.7 %
Percent GA4-available: 4.2 %


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Out of 104 total clients, only 26 clients have GA4 access at all , the rest, 53 clients, have no GA4 tracking connected. On top of that, even for the clients that do have GA4, tracking only starts as early as October 2025, so history is short.

This means my full engagement score (which needs both CTR and engagement rate) can only really be trusted for around a quarter of my clients. For the remaining clients, I only have GSC data, so I can only score them on CTR, not full engagement.

Because of this, I am scoping my engagement scoring to clients who have GA4 access. For the rest, only a CTR-based signal will be available, and I will be upfront about this gap instead of pretending the score means the same thing for everyone.

In [6]:
# ============================================================
# BACKUP CODE FOR PART 4 LIMITATION: GA4/GSC Coverage Gap
# ============================================================

total_clients = df_clients.shape[0]

# Claim: "26 clients have GA4 access at all, 53 have none"
ga4_missing = df_clients["ga4_data_start"].isna().sum()
ga4_present = total_clients - ga4_missing

# Claim: "52 clients have GSC access"
gsc_missing = df_clients["gsc_data_start"].isna().sum()
gsc_present = total_clients - gsc_missing

# Claim: "GA4 tracking only starts as early as October 2025"
earliest_ga4_start = df_clients["ga4_data_start"].dropna().min()

# Claim: "trusted for around a quarter of clients"
pct_ga4_available = round((ga4_present / total_clients) * 100, 1)

print("Total clients:", total_clients)
print(f"Clients WITH GA4 access: {ga4_present}")
print(f"Clients WITH NO GA4 access at all: {ga4_missing}")
print(f"Clients WITH GSC access: {gsc_present}")
print(f"Earliest GA4 start date across all clients: {earliest_ga4_start}")
print(f"Percent of clients with usable GA4 data: {pct_ga4_available}%")

Total clients: 104
Clients WITH GA4 access: 51
Clients WITH NO GA4 access at all: 53
Clients WITH GSC access: 67
Earliest GA4 start date across all clients: 2025-10-29
Percent of clients with usable GA4 data: 49.0%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.